In [0]:
# Limpieza de widgets previos al inicio del proceso para evitar conflictos con ejecuciones anteriores
dbutils.widgets.removeAll()

In [0]:
# Importación de librerías de PySpark para transformaciones y funciones con alias explícito (F)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# Parámetros de entrada: catálogo, esquema origen (silver) y destino (golden)
dbutils.widgets.text("catalogo", "cat_ftr_smartdata_dev")
dbutils.widgets.text("source", "silver")
dbutils.widgets.text("sink", "golden")

In [0]:
# Recuperación de los valores de los widgets para usarlos como variables en el proceso
catalogo = dbutils.widgets.get("catalogo")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

## Capa Gold — Ventas por Categoría y Mes
Agrega ventas a nivel **categoría + mes** con KPIs de volumen, ingresos, descuentos y mix de tipo de venta. Incluye TRIMESTRE para análisis trimestral en Power BI.

In [0]:
# Lectura de la tabla Silver enriquecida como fuente de la capa Gold
# ANIO, MES, TRIMESTRE, IMPORTE_BRUTO e IMPORTE_NETO ya están pre-calculados en Silver
df_silver = spark.table(f"{catalogo}.{schema_source}.ventas_productos_categorias")
#display(df_silver.limit(5))

## Agregación: KPIs mensuales por categoría

In [0]:
# Agregación mensual por categoría usando importes pre-calculados de Silver
# ANIO, MES y TRIMESTRE ya están en Silver; no se recalculan desde FECHA
# ETIQUETA_DESCUENTO usa valores normalizados de Silver: "Precio Normal", "Promoción", "Bono Empresarial", "Regalo"
df_ventas_categoria = (
    df_silver
    .groupBy(
        F.col("CATEGORIA"),
        F.col("ANIO"),
        F.col("MES"),
        F.col("TRIMESTRE")
    )
    .agg(
        # — Volumen —
        F.countDistinct("ID_PEDIDO").alias("NUM_PEDIDOS"),
        F.countDistinct("ID_ARTICULO").alias("NUM_ARTICULOS"),
        F.sum("CANTIDAD").alias("TOTAL_UNIDADES"),
        F.count("*").alias("NUM_LINEAS_TOTAL"),

        # — Ingresos (campos pre-calculados en Silver) —
        F.round(F.sum("IMPORTE_BRUTO"), 2).alias("INGRESO_BRUTO"),
        F.round(F.sum("IMPORTE_NETO"),  2).alias("INGRESO_NETO"),
        F.round(F.avg("PRECIO"),         2).alias("PRECIO_PROMEDIO"),

        # — Descuento —
        F.count(
            F.when(F.col("DESCUENTO") > 0, 1)
        ).alias("NUM_LINEAS_CON_DESCUENTO"),
        F.round(
            F.avg(F.when(F.col("DESCUENTO") > 0, F.col("DESCUENTO"))), 2
        ).alias("DESCUENTO_MEDIO_PCT"),

        # — Etiquetas de descuento (valores normalizados de Silver) —
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Precio Normal",    1)).alias("NUM_PRECIO_NORMAL"),
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Promoción",        1)).alias("NUM_PROMOCION"),
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Bono Empresarial", 1)).alias("NUM_BONO_EMPRESARIAL"),
        F.count(F.when(F.col("ETIQUETA_DESCUENTO") == "Regalo",           1)).alias("NUM_REGALO")
    )
    .withColumn(
        "TICKET_PROMEDIO",
        F.round(F.col("INGRESO_NETO") / F.col("NUM_PEDIDOS"), 2)
    )
    .withColumn(
        "PCT_LINEAS_CON_DESCUENTO",
        F.round(
            F.col("NUM_LINEAS_CON_DESCUENTO") / F.col("NUM_LINEAS_TOTAL") * 100, 2
        )
    )
    # — Auditoría Gold —
    .withColumn("FECHA_ACTUALIZACION", F.current_timestamp())  # Cuándo se calculó la agregación
    .withColumn("INGESTION_USER",     F.current_user())        # Quién ejecutó el pipeline Gold
)

#display(df_ventas_categoria.orderBy("CATEGORIA", "ANIO", "MES").limit(10))

## Escritura en capa Gold

In [0]:
# Escritura en Gold con overwriteSchema para soportar la evolución del esquema
target_table = f"{catalogo}.{schema_sink}.ventas_categoria_mes"

(
    df_ventas_categoria
    .select(
        F.col("CATEGORIA").cast("string"),
        F.col("ANIO").cast("int"),
        F.col("MES").cast("int"),
        F.col("TRIMESTRE").cast("int"),
        F.col("NUM_PEDIDOS").cast("int"),
        F.col("NUM_ARTICULOS").cast("int"),
        F.col("TOTAL_UNIDADES").cast("int"),
        F.col("NUM_LINEAS_TOTAL").cast("int"),
        F.col("INGRESO_BRUTO").cast("double"),
        F.col("INGRESO_NETO").cast("double"),
        F.col("TICKET_PROMEDIO").cast("double"),
        F.col("PRECIO_PROMEDIO").cast("double"),
        F.col("NUM_LINEAS_CON_DESCUENTO").cast("int"),
        F.col("PCT_LINEAS_CON_DESCUENTO").cast("double"),
        F.col("DESCUENTO_MEDIO_PCT").cast("double"),
        F.col("NUM_PRECIO_NORMAL").cast("int"),
        F.col("NUM_PROMOCION").cast("int"),
        F.col("NUM_BONO_EMPRESARIAL").cast("int"),
        F.col("NUM_REGALO").cast("int"),
        F.col("FECHA_ACTUALIZACION").cast("timestamp"),
        F.col("INGESTION_USER").cast("string"),
    )
    .coalesce(4)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"✓ Tabla '{target_table}' escrita correctamente en capa Gold.")

## Verificación

In [0]:
%sql
-- Verificación de ventas_categoria_mes incluyendo TRIMESTRE y etiquetas de descuento corregidas
SELECT * FROM cat_ftr_smartdata_dev.golden.ventas_categoria_mes
ORDER BY CATEGORIA, ANIO, MES
LIMIT 10